In [ ]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append("../../../")

import datetime, math, warnings
import numpy as np
import pandas as pd
import QuantLib as ql
import pytz
import matplotlib.pyplot as plt
import matplotlib.pylab as pylab
plt.style.use('ggplot')
pylab.rcParams.update({'figure.figsize': (18, 10), 'axes.titlesize': 'large', 'axes.labelsize': 'medium', 'xtick.labelsize': 'medium', 'ytick.labelsize': 'medium'})

NYC_tz = pytz.timezone("America/New_York")

from MDP.IRSwaps.IRSwapsMDP import IRSwapsMDP
from MDP.IRSwaps.SDR_INTRADAY.rl_curve_utils.stir_curve_building_utils import get_fomc_meetings_list
from MDP.STIRFutures.STIRFutureOptionMDP import STIRFutureOptionMDP
from Query.IRSwaps.IRSwapQuery import IRSwapQuery
from RVUtils.ImpliedDistribution import SFRImpliedDistribution

In [ ]:
AS_OF = datetime.date(2026, 5, 22)
CURVE = "USD-SOFR-1D-Q12STIRT"
MIN_INFORMATIVE_STD_BPS = 25.0

IMM_TENORS = [
    "IMM_1xIMM_2", "IMM_2xIMM_3", "IMM_3xIMM_4", "IMM_4xIMM_5",
    "IMM_5xIMM_6", "IMM_6xIMM_7", "IMM_7xIMM_8", "IMM_8xIMM_9",
    "IMM_9xIMM_10", "IMM_10xIMM_11", "IMM_11xIMM_12", "IMM_12xIMM_13",
]
IMM_MONTH_MAP = {"H": 3, "M": 6, "U": 9, "Z": 12}

def imm_code_to_sfr(code):
    return f"SFR{code[0]}2{code[1]}"

def imm_to_approx_dates(code):
    m = IMM_MONTH_MAP[code[0]]
    y = 2020 + int(code[1])
    start = datetime.date(y, m, 15)
    em = m + 3
    ey = y
    if em > 12: em -= 12; ey += 1
    return start, datetime.date(ey, em, 15)

def count_fomc(start, end, fomc_list):
    return sum(1 for d in fomc_list if start <= d < end)

## Fetch curve, compute spreads and butterflies

In [ ]:
ts = NYC_tz.localize(datetime.datetime.combine(AS_OF, datetime.time(17, 0)))
curve_mdp = IRSwapsMDP(source="BARCHART_STIRF-RL")
curve_handle = curve_mdp.get_pricer(request=dict(curve_name=CURVE, timestamp=ts))

contracts, prices, rates = [], [], []
for t in IMM_TENORS:
    q = IRSwapQuery(curve=CURVE, tenor=t).resolve_query(ts, pricer_or_curve=curve_handle)
    pkg, _ = q.resolve_package(pricer_or_curve=curve_handle)
    eff = pkg[0].__dict__["kwargs"]["effective"]
    price = 100 - pkg[0].__dict__["kwargs"]["fixed_rate"]
    rate = pkg[0].__dict__["kwargs"]["fixed_rate"] * 100
    imm = ql.IMM.code(ql.Date(eff.day, eff.month, eff.year))
    contracts.append(imm)
    prices.append(price)
    rates.append(rate)

cal_spreads = [(prices[i] - prices[i+1]) * 10000 for i in range(len(contracts)-1)]

butterflies = []
for i in range(len(contracts) - 2):
    near, mid, far = contracts[i], contracts[i+1], contracts[i+2]
    butterflies.append({
        "name": f"BF {near}-{mid}-{far}",
        "near": near, "mid": mid, "far": far,
        "near_sym": imm_code_to_sfr(near), "mid_sym": imm_code_to_sfr(mid), "far_sym": imm_code_to_sfr(far),
        "bf_bps": (prices[i] - 2*prices[i+1] + prices[i+2]) * 10000,
        "sp_near": cal_spreads[i], "sp_far": cal_spreads[i+1],
    })

# FOMC overlay
fomc_raw = get_fomc_meetings_list(as_of=AS_OF, n_plus_years=2)
fomc_dates = [d.date() if isinstance(d, datetime.datetime) else d for d in fomc_raw]
fomc_dates = [d for d in fomc_dates if d > AS_OF]

for bf in butterflies:
    for leg in ["near", "mid", "far"]:
        s, e = imm_to_approx_dates(bf[leg])
        bf[f"fomc_{leg}"] = count_fomc(s, e, fomc_dates)

print(f"Strip: {' -> '.join(contracts)}")
print(f"Rates: {min(rates):.2f}% - {max(rates):.2f}%")

## Fetch SABR smiles & extract BKM + BL moments

In [ ]:
stirfo_mdp = STIRFutureOptionMDP(source="BARCHART_STIRFO-QL")
dist = SFRImpliedDistribution(use_sabr_vols=True, sabr_extrapolation=True, sabr_n_strikes=200)

moment_cache = {}
all_symbols = sorted(set(s for bf in butterflies for s in [bf["near_sym"], bf["mid_sym"], bf["far_sym"]]))

for sym in all_symbols:
    smile = None
    for kwargs in [
        {"symbol": sym, "as_of": AS_OF, "show_tqdm": False},
        {"symbol": sym, "as_of": AS_OF, "strike_offsets_bps": "listed", "show_tqdm": False},
    ]:
        try:
            smile = stirfo_mdp.fetch_sabr_smile(kwargs)
            break
        except Exception:
            continue
    if smile is None:
        print(f"  {sym}: FAILED")
        moment_cache[sym] = None
        continue
    try:
        snap = dist.extract(smile, run_bl=True, run_gm=False, run_bkm=True)
        moment_cache[sym] = {"bkm": snap.bkm_result, "bl": snap.bl_result, "fwd": smile.params.forward_rate, "tte": smile.params.time_to_expiry}
        bkm = snap.bkm_result
        print(f"  {sym}: fwd={smile.params.forward_rate:.4f}%  BKM_std={bkm.std_rate*100:.1f}bps  skew={bkm.skewness_rate:+.3f}  TTE={int(smile.params.time_to_expiry*365)}d")
    except Exception as e:
        print(f"  {sym}: extraction failed - {e}")
        moment_cache[sym] = None

## Variance Term Structure

Under a constant-vol diffusion, std grows as σ√T — the sqrt(T) benchmark curve.  
Deviations above the curve mean the contract has **excess uncertainty** beyond what time alone explains.  
Deviations below mean **deficit uncertainty** — the market is more confident than the smooth curve implies.  

When the middle leg of a butterfly sits above the curve more than its wings, the fly is variance-rich.

In [ ]:
# Build term structure DataFrame
ts_rows = []
for sym in all_symbols:
    m = moment_cache.get(sym)
    if m and m["bkm"]:
        ts_rows.append({
            "symbol": sym,
            "contract": sym.replace("SFR", ""),
            "tte": m["tte"],
            "tte_days": int(m["tte"] * 365),
            "fwd_rate": m["fwd"],
            "bkm_std": m["bkm"].std_rate * 100,
            "bl_std": m["bl"].std_rate * 100 if m["bl"] else np.nan,
            "skew": m["bkm"].skewness_rate,
            "kurt": m["bkm"].kurtosis_rate,
        })

ts_df = pd.DataFrame(ts_rows).sort_values("tte").reset_index(drop=True)

# Fit sqrt(T) benchmark: find sigma that best fits std = sigma * sqrt(T)
# Use OLS: sigma = sum(std_i * sqrt(T_i)) / sum(T_i)
mask = ts_df["tte"] > 0.1  # exclude near-dead contracts from fit
sigma_fit = (ts_df.loc[mask, "bkm_std"] * np.sqrt(ts_df.loc[mask, "tte"])).sum() / ts_df.loc[mask, "tte"].sum()

tte_grid = np.linspace(0.01, ts_df["tte"].max() * 1.05, 200)
sqrt_benchmark = sigma_fit * np.sqrt(tte_grid)
ts_df["sqrt_benchmark"] = sigma_fit * np.sqrt(ts_df["tte"])
ts_df["excess_vs_sqrt"] = ts_df["bkm_std"] - ts_df["sqrt_benchmark"]

print(f"Fitted implied vol (sigma): {sigma_fit:.1f} bps/sqrt(yr)")
display(ts_df[["contract", "tte_days", "fwd_rate", "bkm_std", "bl_std", "sqrt_benchmark", "excess_vs_sqrt", "skew"]].round(2))

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(18, 16), gridspec_kw={"height_ratios": [3, 1.5, 1.5]})

# ── Panel 1: Variance Term Structure ──────────────────────────────────────────
ax = axes[0]
ax.plot(tte_grid, sqrt_benchmark, 'k--', alpha=0.5, linewidth=1.5, label=f'$\\sigma\\sqrt{{T}}$ benchmark ($\\sigma$={sigma_fit:.0f}bps/$\\sqrt{{yr}}$)')
ax.plot(ts_df["tte"], ts_df["bkm_std"], 'o-', color='#1f77b4', markersize=8, linewidth=2, label='BKM std (model-free)', zorder=5)
ax.plot(ts_df["tte"], ts_df["bl_std"], 's--', color='#ff7f0e', markersize=5, linewidth=1, alpha=0.6, label='BL std (spline derivative)')

# Shade excess / deficit vs sqrt(T)
for _, row in ts_df.iterrows():
    color = '#d62728' if row["excess_vs_sqrt"] > 0 else '#2ca02c'
    ax.vlines(row["tte"], row["sqrt_benchmark"], row["bkm_std"], color=color, linewidth=2.5, alpha=0.7)

for _, row in ts_df.iterrows():
    offset = 3 if row["excess_vs_sqrt"] >= 0 else -6
    ax.annotate(row["contract"], (row["tte"], row["bkm_std"]),
                textcoords="offset points", xytext=(0, offset),
                ha='center', fontsize=9, fontweight='bold')

ax.set_ylabel('Implied Std Dev (bps)')
ax.set_title(f'SOFR Options Variance Term Structure — {AS_OF}\n'
             f'Red bars = excess uncertainty (variance-rich) | Green bars = deficit (variance-cheap)',
             fontsize=13)
ax.legend(loc='upper left', fontsize=10)
ax.set_xlim(left=0)
ax.set_ylim(bottom=0)

# ── Panel 2: Excess vs sqrt(T) benchmark (the butterfly signal source) ────────
ax2 = axes[1]
colors = ['#d62728' if v > 0 else '#2ca02c' for v in ts_df["excess_vs_sqrt"]]
bars = ax2.bar(range(len(ts_df)), ts_df["excess_vs_sqrt"], color=colors, alpha=0.8, edgecolor='black', linewidth=0.5)
ax2.set_xticks(range(len(ts_df)))
ax2.set_xticklabels(ts_df["contract"], fontsize=9)
ax2.axhline(0, color='black', linewidth=0.8)
ax2.set_ylabel('Excess vs $\\sigma\\sqrt{T}$ (bps)')
ax2.set_title('Per-Contract Variance Excess — where butterflies get their signal', fontsize=11)

for i, (_, row) in enumerate(ts_df.iterrows()):
    val = row["excess_vs_sqrt"]
    offset = 0.3 if val >= 0 else -0.3
    ax2.text(i, val + offset, f'{val:+.1f}', ha='center', fontsize=8, fontweight='bold')

# ── Panel 3: Implied vol (std / sqrt(T)) — the detrended view ────────────────
ax3 = axes[2]
impl_vol = ts_df["bkm_std"] / np.sqrt(ts_df["tte"].clip(lower=0.01))
ax3.plot(range(len(ts_df)), impl_vol, 'D-', color='#9467bd', markersize=7, linewidth=2)
ax3.axhline(sigma_fit, color='black', linewidth=1, linestyle='--', alpha=0.5, label=f'Fitted $\\sigma$={sigma_fit:.0f}')
ax3.set_xticks(range(len(ts_df)))
ax3.set_xticklabels(ts_df["contract"], fontsize=9)
ax3.set_ylabel('Implied Vol (bps/$\\sqrt{yr}$)')
ax3.set_title('Detrended: Implied Vol = Std / $\\sqrt{T}$ — flat means variance term structure is "normal"', fontsize=11)
ax3.legend(loc='upper right', fontsize=9)

for i, v in enumerate(impl_vol):
    ax3.text(i, v + 2, f'{v:.0f}', ha='center', fontsize=8)

plt.tight_layout()
plt.show()

## Compute screener layers

In [ ]:
for bf in butterflies:
    m_n = moment_cache.get(bf["near_sym"])
    m_m = moment_cache.get(bf["mid_sym"])
    m_f = moment_cache.get(bf["far_sym"])

    if all(m and m.get("bkm") for m in [m_n, m_m, m_f]):
        std_n = m_n["bkm"].std_rate * 100
        std_m = m_m["bkm"].std_rate * 100
        std_f = m_f["bkm"].std_rate * 100
        tte_n, tte_m, tte_f = max(m_n["tte"], 1e-6), max(m_m["tte"], 1e-6), max(m_f["tte"], 1e-6)

        # Raw linear interp
        bf["std_n"], bf["std_m"], bf["std_f"] = std_n, std_m, std_f
        bf["std_excess"] = std_m - (std_n + std_f) / 2

        # sqrt(T)-adjusted interp
        vol_n = std_n / math.sqrt(tte_n)
        vol_f = std_f / math.sqrt(tte_f)
        w = (tte_m - tte_n) / max(tte_f - tte_n, 1e-6)
        vol_interp = vol_n * (1 - w) + vol_f * w
        std_adj_interp = vol_interp * math.sqrt(tte_m)
        bf["adj_excess"] = std_m - std_adj_interp

        fly_abs = max(abs(bf["bf_bps"]), 0.5)
        bf["var_signal"] = bf["std_excess"] / fly_abs
        bf["adj_signal"] = bf["adj_excess"] / fly_abs

        bf["near_dead"] = std_n < MIN_INFORMATIVE_STD_BPS

        # Layer 2: probability space
        bf["kw"] = abs(bf["bf_bps"]) / max(std_m, 1.0)

        # Layer 3: fragility
        rng = [4 * s for s in [std_n, std_m, std_f]]
        deltas = [0.01 * r for r in rng]
        bf["fragility"] = abs(deltas[0] - 2*deltas[1] + deltas[2]) / fly_abs
    else:
        for k in ["std_n","std_m","std_f","std_excess","adj_excess","var_signal","adj_signal","kw","fragility"]:
            bf[k] = np.nan
        bf["near_dead"] = False

## Composite Results

In [ ]:
rows = []
for bf in butterflies:
    rows.append({
        "Butterfly": bf["name"],
        "Level": round(bf["bf_bps"], 2),
        "Prob chg (pp)": round(bf["bf_bps"] / 25 * 100, 0),
        "Std N": round(bf.get("std_n", np.nan), 0),
        "Std M": round(bf.get("std_m", np.nan), 0),
        "Std F": round(bf.get("std_f", np.nan), 0),
        "Raw Excess": round(bf.get("std_excess", np.nan), 1),
        "Adj Excess": round(bf.get("adj_excess", np.nan), 1),
        "Raw Sig": round(bf.get("var_signal", np.nan), 2),
        "Adj Sig": round(bf.get("adj_signal", np.nan), 2),
        "K/W": round(bf.get("kw", np.nan), 3),
        "Fragility": round(bf.get("fragility", np.nan), 3),
        "FOMC": f"{bf.get('fomc_near',0)}/{bf.get('fomc_mid',0)}/{bf.get('fomc_far',0)}",
        "Near Dead": bf.get("near_dead", False),
    })

results_df = pd.DataFrame(rows).set_index("Butterfly")

def highlight_signals(row):
    styles = [''] * len(row)
    adj_sig_idx = row.index.get_loc('Adj Sig')
    adj_exc_idx = row.index.get_loc('Adj Excess')
    if pd.notna(row['Adj Sig']):
        if row['Near Dead']:
            styles[adj_sig_idx] = 'color: gray; text-decoration: line-through'
            styles[adj_exc_idx] = 'color: gray; text-decoration: line-through'
        elif row['Adj Sig'] > 0.3:
            styles[adj_sig_idx] = 'background-color: #ffcccc; font-weight: bold'
            styles[adj_exc_idx] = 'background-color: #ffcccc'
        elif row['Adj Sig'] < -0.3:
            styles[adj_sig_idx] = 'background-color: #ccffcc; font-weight: bold'
            styles[adj_exc_idx] = 'background-color: #ccffcc'
    return styles

display(results_df.style.apply(highlight_signals, axis=1).format(precision=2, na_rep='-'))

## Butterfly Decomposition: Raw vs Time-Adjusted

In [ ]:
bf_names = [bf["name"].replace("BF ", "") for bf in butterflies]
raw_excess = [bf.get("std_excess", np.nan) for bf in butterflies]
adj_excess = [bf.get("adj_excess", np.nan) for bf in butterflies]
bf_levels = [bf["bf_bps"] for bf in butterflies]

fig, axes = plt.subplots(2, 1, figsize=(16, 10))

# Panel 1: Raw vs Adjusted excess side by side
ax = axes[0]
x = np.arange(len(bf_names))
w = 0.35
bars1 = ax.bar(x - w/2, raw_excess, w, label='Raw Std Excess (linear interp)', color='#1f77b4', alpha=0.7, edgecolor='black', linewidth=0.5)
bars2 = ax.bar(x + w/2, adj_excess, w, label='Adj Std Excess ($\\sqrt{T}$ interp)', color='#d62728', alpha=0.7, edgecolor='black', linewidth=0.5)
ax.set_xticks(x)
ax.set_xticklabels(bf_names, fontsize=9)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_ylabel('Std Excess at Middle Leg (bps)')
ax.set_title('Raw vs Time-Adjusted Variance Excess per Butterfly\n'
             'Bars that shrink/flip after adjustment = mechanical signal from sqrt(T) curvature', fontsize=12)
ax.legend(fontsize=10)

for i in range(len(bf_names)):
    if not np.isnan(raw_excess[i]):
        ax.text(i - w/2, raw_excess[i] + 0.2, f'{raw_excess[i]:+.1f}', ha='center', fontsize=7)
    if not np.isnan(adj_excess[i]):
        ax.text(i + w/2, adj_excess[i] + 0.2, f'{adj_excess[i]:+.1f}', ha='center', fontsize=7)

# Panel 2: Butterfly levels with adjusted signal coloring
ax2 = axes[1]
adj_sigs = [bf.get("adj_signal", np.nan) for bf in butterflies]
colors = []
for i, s in enumerate(adj_sigs):
    if butterflies[i].get("near_dead"):
        colors.append('gray')
    elif np.isnan(s):
        colors.append('lightgray')
    elif s > 0.3:
        colors.append('#d62728')  # red = rich
    elif s < -0.3:
        colors.append('#2ca02c')  # green = cheap
    else:
        colors.append('#1f77b4')  # blue = fair

ax2.bar(x, bf_levels, color=colors, edgecolor='black', linewidth=0.5, alpha=0.8)
ax2.set_xticks(x)
ax2.set_xticklabels(bf_names, fontsize=9)
ax2.axhline(0, color='black', linewidth=0.8)
ax2.set_ylabel('Butterfly Level (bps)')
ax2.set_title('Butterfly Levels colored by Adj Variance Signal\n'
             'Red = variance-rich (sell) | Green = variance-cheap (buy) | Blue = fair | Gray = near-expiry dead leg', fontsize=12)

for i in range(len(bf_names)):
    offset = 0.2 if bf_levels[i] >= 0 else -0.4
    ax2.text(i, bf_levels[i] + offset, f'{bf_levels[i]:+.1f}', ha='center', fontsize=8, fontweight='bold')

plt.tight_layout()
plt.show()

## Actionable Signals

In [ ]:
for bf in butterflies:
    signals, caveats = [], []
    vs_adj = bf.get("adj_signal", np.nan)
    vs_raw = bf.get("var_signal", np.nan)
    frag = bf.get("fragility", np.nan)
    kw = bf.get("kw", np.nan)

    if not np.isnan(vs_adj) and abs(vs_adj) > 0.3:
        if bf.get("near_dead"):
            caveats.append(f"L1 SUPPRESSED: near leg dead (std={bf.get('std_n',0):.0f}bps)")
        else:
            direction = "RICH (sell)" if vs_adj > 0 else "CHEAP (buy)"
            signals.append(f"L1 Variance: {direction} (adj_excess={bf.get('adj_excess',0):+.1f}bps, adj_signal={vs_adj:+.2f})")
    elif not np.isnan(vs_raw) and abs(vs_raw) > 0.3 and (np.isnan(vs_adj) or abs(vs_adj) <= 0.3):
        caveats.append(f"L1 raw ({vs_raw:+.2f}) absorbed by sqrt(T) -> adj={vs_adj:+.2f}")

    if not np.isnan(kw) and kw > 0.15:
        signals.append(f"L2 High timing conviction (K/W={kw:.3f})")
    elif not np.isnan(kw) and kw < 0.03 and abs(bf["bf_bps"]) > 1.0:
        signals.append(f"L2 Weak kink vs uncertainty (K/W={kw:.3f})")

    if not np.isnan(frag) and frag > 0.5:
        signals.append(f"L3 HIGH fragility ({frag:.3f}) -> size down")
    elif not np.isnan(frag) and frag < 0.15:
        signals.append(f"L3 LOW fragility ({frag:.3f}) -> full size OK")

    fomc_asym = abs(bf.get("fomc_near", 0) - bf.get("fomc_far", 0))
    if fomc_asym >= 1:
        signals.append(f"FOMC: asymmetric ({bf['fomc_near']}/{bf['fomc_mid']}/{bf['fomc_far']})")

    if signals or caveats:
        print(f"\n{bf['name']} ({bf['bf_bps']:+.2f} bps):")
        for s in signals:
            print(f"  + {s}")
        for c in caveats:
            print(f"  ~ {c}")